# Vector-Quantized Variational Autoencoder (VQ-VAE)

Source:

  https://github.com/MishaLaskin/vqvae

Adapted:

  Antonio Esteves @ UMinho, April 2024

## Import necessary libraries

In [ ]:
import yaml
import wandb
import os
import random
import numpy             as     np
import pandas            as     pd
import matplotlib.pyplot as     plt
from   pathlib           import Path
from   PIL               import Image
import torch
from   torch.utils.data  import Dataset
from   torch.utils.data  import DataLoader
from   torchvision       import transforms, utils
from   natsort           import natsorted
import time

import torch.nn            as nn
import torch.nn.functional as F
from   tqdm.notebook       import trange, tqdm
from   torchvision.utils   import make_grid
from   typing              import Tuple, Dict, List

## Import W&B and Login

In [ ]:
wandb.login()

## Read the configuration file

In [ ]:
calculate_statistics      = False
LOAD_VQVAE_MODEL          = True
SKIP_TRAIN_VQVAE_MODEL    = True
LOAD_PIXELCNN_MODEL       = False
SKIP_TRAIN_PIXELCNN_MODEL = False
SKIP_COMPUTE_CODEBOOK     = True
sample_size               = 8

CONFIG_FILE               = 'config/config4.yaml'
with open(CONFIG_FILE, 'r') as file:
    try:
        config = yaml.safe_load(file)
    except yaml.YAMLError as exc:
        print(exc)

BASE_FILE_NAME = config["exp_params"]["vqvae_save_file"]
config["exp_params"]["learning_rate"] = float(config["exp_params"]["learning_rate"])
config["exp_params"]["pixelcnn_learning_rate"] = \
    float(config["exp_params"]["pixelcnn_learning_rate"])

In [ ]:
print('model_params:')
model_params = config['model_params']
for key, value in model_params.items():
    print(f'\t{key}: {value}')

print('exp_params:')
exp_params = config['exp_params']
for key, value in exp_params.items():
    print(f'\t{key}: {value}')

## Track metadata and hyperparameters with `wandb.init`

Define the experiment: the hyperparameters, the dataset and model name. This information will be stored in a `config` dictionary.

In [ ]:
config_wandb = dict(
    model                  = config["model_params"]["name"],
    config                 = CONFIG_FILE,
    VQVAE_file             = f'{config["exp_params"]["vqvae_save_file"]}.pth',
    PixelCNN_file          = f'{config["exp_params"]["pixelcnn_save_file"]}.pth',
    encoder_hidden_dim     = config["model_params"]["encoder_hidden_dim"],
    residual_hidden_dim    = config["model_params"]["residual_hidden_dim"],
    num_residual_layers    = config["model_params"]["num_residual_layers"],
    num_stride2            = config["model_params"]["num_stride2"],
    embedding_dim          = config["model_params"]["embedding_dim"],
    embedding_number       = config["model_params"]["embedding_number"],
    pixelcnn_hidden_dim    = config["model_params"]["pixelcnn_hidden_channels"],
    gated_layers           = config["model_params"]["gated_layers"],
    patch_size             = config["model_params"]["patch_size"],
    num_channels           = config["model_params"]["num_channels"],
    beta                   = config["model_params"]["beta"],
    dataset                = config["exp_params"]["dataset"],
    train_batch_size       = config["exp_params"]["train_batch_size"],
    val_batch_size         = config["exp_params"]["val_batch_size"],
    test_batch_size        = config["exp_params"]["test_batch_size"],
    epochs                 = config["exp_params"]["num_epochs"],
    learning_rate          = config["exp_params"]["learning_rate"],
    optimizer              = config["exp_params"]["optimizer"],
    scheduler              = config["exp_params"]["vqvae_scheduler"],
    pixelcnn_learning_rate = config["exp_params"]["pixelcnn_learning_rate"],
    pixelcnn_optimizer     = config["exp_params"]["pixelcnn_optimizer"],
    pixelcnn_scheduler     = config["exp_params"]["pixelcnn_scheduler"],
    pixelcnn_epochs        = config["exp_params"]["pixelcnn_num_epochs"],
    log_interval           = config["exp_params"]["log_interval"],
)

In [ ]:
print('W&B saved parameters:')
for key, value in config_wandb.items():
    print(f'\t{key}: {value}')

In [ ]:
wandb.init(project='VQVAE_CelebA', entity='ajesteves', config=config_wandb)

## Initializations

In [ ]:
print(f'Torch version {torch.__version__}')

# setup computing device agnostic code

device = "cuda" if torch.cuda.is_available() else "cpu"

print(f'Using {device} as computing device')

# Setup path to celebA dataset folder

data_path = Path(config["exp_params"]["data_path"])

train_dir = data_path / "train"
val_dir   = data_path / "val"
test_dir  = data_path / "test"

## Exploring the CelebA dataset

In [ ]:
# Get all training image paths
train_image_list = list(train_dir.glob("*.jpg"))

print(f'Training set size: {len(train_image_list)}')

# Get all validation image paths
val_image_list = list(val_dir.glob("*.jpg"))

print(f'Validation set size: {len(val_image_list)}')

# Get all test image paths
test_image_list = list(test_dir.glob("*.jpg"))

print(f'Test set size: {len(test_image_list)}')

### Visualizing random images

In [ ]:
# set seed

random.seed(42)

# Pick a random image path

rand_image_path = random.choice(train_image_list)
print(f'Randomly selected image:\n\t{rand_image_path}')

# Open the image using pillow library

img = Image.open(rand_image_path)

# Show the image and print image metadata

print(f'Random image path:   {rand_image_path}')
print(f'Random image height: {img.height}')
print(f'Random image width:  {img.width}')

display(img)

In [ ]:
# Visualize an image using matplotlib

# convert the image 'img' to a numpy array

img_array = np.asarray(img)

# plot the image with matplotlib

plt.figure(figsize=(10,7))
plt.imshow(img_array)
plt.title(f'Image shape: {img_array.shape} (height,width,channels)')
plt.axis(False)

## Transforming the Data

Before we can use our data with PyTorch:
1. Convert the images to tensors using `torchvision.transforms`([documentation](https://pytorch.org/vision/0.16//transforms.html))
2. Create a custom Dataset from the images in a folder and associated attributes using `torch.utils.data.Dataset`
3. Create training, validation and test datasets
4. Convert the `Datasets` into `torch.utils.data.DataLoaders`

### (i) Convert images to tensors

In [ ]:
train_transforms = transforms.Compose(
    [
    #transforms.RandomHorizontalFlip(),
    transforms.Resize(config["model_params"]["patch_size"]),
    transforms.CenterCrop(config["model_params"]["patch_size"]),
    transforms.ToTensor()
    ]
)

val_transforms = transforms.Compose(
    [
    #transforms.RandomHorizontalFlip(),
    transforms.Resize(config["model_params"]["patch_size"]),
    transforms.CenterCrop(config["model_params"]["patch_size"]),
    transforms.ToTensor()
    ]
)

In [ ]:
img_tensor = train_transforms(img)

print(img_tensor.shape)
print(img_tensor)

### Visualize transformed images

In [ ]:
def plot_transformed_images(
    image_paths,
    transform,
    n           = 3,
    seed        = None ):
  """
  The functions selects 'n' random images from 'image_paths', load and transform them,
  then it plots the original images vs the transformed representation.
  """
  # set the random seed if it is not None
  if seed:
    random.seed(seed)

  # randomly select 'n' image paths
  rand_image_paths = random.sample(image_paths,k=n)

  # iterate throuth the 'n' selected paths
  for image_path in rand_image_paths:

    with Image.open(image_path) as f:

      # original image
      fig,ax = plt.subplots(nrows=1,ncols=2)
      ax[0].imshow(f)
      ax[0].set_title(f'Original\nSize: {f.size}')
      ax[0].axis(False)

      # transform the original image and plot the transformed representation
      transformed_image = transform(f).permute(1, 2, 0) # we need to convert C,H,W (pytorch) to H,W,C (plt)
      ax[1].imshow(transformed_image)
      ax[1].set_title(f'Transformed\nShape: {transformed_image.shape}')
      ax[1].axis(False)

      fig.suptitle(f'Class: {image_path.parent.stem}',fontsize=16)

In [ ]:
plot_transformed_images(
    image_paths = train_image_list,
    transform   = train_transforms,
    n           = 3
)

### (ii) Create a custom Dataset from the images in a folder and the associated attributes using `torch.utils.data.Dataset`

In [ ]:
class CustomDataSet(Dataset):

    def __init__(self, root_dir, transform):
        self.root_dir     = root_dir
        self.transform    = transform
        self.all_images   = os.listdir(root_dir)
        self.total_images = natsorted(self.all_images)

    def __len__(self):
        return len(self.total_images)

    def __getitem__(self, idx):
        img_loc = os.path.join(self.root_dir, self.total_images[idx])
        image   = Image.open(img_loc).convert("RGB")
        tensor_image = self.transform(image)
        return tensor_image

In [ ]:
# Create training and validation datasets

train_data = CustomDataSet(
    train_dir,
    transform = train_transforms
)

val_data = CustomDataSet(
    val_dir,
    transform = val_transforms
)

test_data = CustomDataSet(
    test_dir,
    transform = val_transforms
)

train_data_len = len(train_data)
val_data_len   = len(val_data)
test_data_len  = len(test_data)
print(train_data_len)
print(val_data_len)
print(test_data_len)

### Visualize some samples from the created training set

In [ ]:
# Indexing the 'train_data' Dataset to get a single image

img  = train_data[0]

print(f'Image tensor:\n{img}')
print(f'Image shape: {img.shape}')
print(f'Image data type: {img.dtype}')

In [ ]:
# rearrange the order of dimensions on image tensor for using matplot lib
img_permuted = img.permute(1, 2, 0) # C,H,W --> H,W,C

# print shapes
print(f'Original image shape (pytorch tensor):   {img.shape}')
print(f'Permute image shape (matplotlib format): {img_permuted.shape}')

# plot the image

plt.figure(figsize=(10,7))
plt.imshow(img_permuted)
plt.axis(False)
plt.title(f'An image from training set')

### (iii) Convert the `Dataset` into a `torch.utils.data.DataLoader`

A DataLoader helps us to iterate trough the dataset em get a batch of samples each time.

Relevant configuration parameters:

```python
config["exp_params"]["train_batch_size"]
config["exp_params"]["val_batch_size"]
config["exp_params"]["test_batch_size"]
config["exp_params"]["num_workers"]
config["exp_params"]["pin_memory"]
```

In [ ]:
# Convert each 'Dataset' into a 'DataLoader'

train_dataloader = DataLoader(
    dataset     = train_data,
    batch_size  = config["exp_params"]["train_batch_size"],
    shuffle     = True,
    num_workers = config["exp_params"]["num_workers"],
    pin_memory  = config["exp_params"]["pin_memory"]
)

val_dataloader = DataLoader(
    dataset     = val_data,
    batch_size  = config["exp_params"]["val_batch_size"],
    shuffle     = False,
    num_workers = config["exp_params"]["num_workers"],
    pin_memory  = config["exp_params"]["pin_memory"]
)

test_dataloader = DataLoader(
    dataset     = test_data,
    batch_size  = config["exp_params"]["test_batch_size"],
    shuffle     = False,
    num_workers = config["exp_params"]["num_workers"],
    pin_memory  = config["exp_params"]["pin_memory"]
)

In [ ]:
print(train_dataloader)
print(val_dataloader)
print(test_dataloader)

print(f'Number of batches in a training epoch:   {len(train_dataloader)}')
print(f'Number of batches in a validation epoch: {len(val_dataloader)}')
print(f'Number of batches in a testing epoch:    {len(test_dataloader)}')

In [ ]:
# print metadata about a sample retrieved from the training DataLoader

imgs = next(iter(train_dataloader))

print(f'Batch of images shape: {imgs.shape}')   # BS, Ch, H, W

### Compute training data variance

In [ ]:
if calculate_statistics == True:
    train_mean = 0.0
    for images in train_dataloader:
        batch_samples = images.size(0)
        images        = images.view(batch_samples, images.size(1), -1)
        train_mean   += images.mean(2).sum(0)
        print('.',end='')
    train_mean = train_mean / len(train_dataloader.dataset)

    train_variance   = 0.0
    pixel_count      = 0
    for images in train_dataloader:
        batch_samples   = images.size(0)
        images          = images.view(batch_samples, images.size(1), -1)
        train_variance += ((images - train_mean.unsqueeze(1))**2).sum([0,2])
        pixel_count    += images.nelement() / images.size(1)
        print('*',end='')
    train_variance /= pixel_count
    train_stddev = torch.sqrt(train_variance)

else:
    train_mean     = torch.tensor([0.5184, 0.4153, 0.3617])
    train_variance = torch.tensor([0.0890, 0.0715, 0.0682])
    train_stddev   = torch.tensor([0.2983, 0.2674, 0.2611])

print(f'Training data mean:     {train_mean}')
print(f'Training data variance: {train_variance}')
print(f'Training data stddev:   {train_stddev}')

## Learning Rate Schedulers

### Cyclic Learning Rate scheduler

Example:

```python
model     = ...
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.CyclicLR(
    optimizer,
    base_lr      = 0.01,         # Initial and minimum learning rate
    max_lr       = 0.1,          # Maximum learning rate
    step_size_up = 5,            # Number of training iterations in the increasing half cycle
    mode         = "triangular", # Can be "triangular" or "triangular2" or "exp_range"
    )

lrs = []
for epoch in range(EPOCHS):
    for i, sample in enumerate(dataloader):

        inputs, labels = ...
        optimizer.zero_grad()
        outputs = model(inputs)
        loss    = loss_fn(outputs, labels)
        loss.backward()
        optimizer.step()

        lrs.append(optimizer.param_groups[0]["lr"])
        scheduler.step()

plt.plot(lrs)
```

### CosineAnnealingWarmRestarts Learning Rate scheduler

The `CosineAnnealingWarmRestarts` is similar to the cosine annealing schedule. However, it allows you to restart the learning rate schedule witt the initial value of the learning rate at each epoch (for example).

Example:

```python
model     = ...
optimizer = torch.optim.Adam(model.parameters(), lr=0.1)
scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer, 
    T_0        = 20,  # Number of iterations in the first restart (cycle), which defines the number of 
                      # iterations since LR_max=lr until LR_min=eta_min.
    T_mult     = 1,   # A factor that multiplies T_0​ after each restart (cycle), which specifies the
                      # number of iterations in next cycle (default=1).
    eta_min    = 1e-4 # The minimum learning rate
    last_epoch = -1,  # When last_epoch=-1, sets initial learning rate as the 'lr' value defined on 'optimizer' 
)

iters = len(dataloader)
lrs   = []

for epoch in range(EPOCHS):
     for i, sample in enumerate(dataloader):

         inputs, labels = ...
         optimizer.zero_grad()
         outputs = model(inputs)
         loss    = loss_fn(outputs, labels)
         loss.backward()
         optimizer.step()

         lrs.append(optimizer.param_groups[0]["lr"])
         scheduler.step(epoch + i / iters)

plt.plot(lrs)
```


### PyTorch learning rate finder

A PyTorch implementation of the learning rate range test detailed in [Cyclical Learning Rates for Training Neural Networks](https://arxiv.org/abs/1506.01186) by Leslie N. Smith and the tweaked version used by `fastai`.

The learning rate range test is a test that provides valuable information about the optimal learning rate. During a pre-training run, the learning rate is increased linearly or exponentially between two boundaries. The low initial learning rate allows the network to start converging and as the learning rate is increased it will eventually be too large and the network will diverge.

Typically, a good static learning rate can be found half-way on the descending loss curve.

For cyclical learning rates (also detailed in Leslie Smith's paper) where the learning rate is cycled between two boundaries (`start_lr`, `end_lr`), the author advises the point at which the loss starts descending and the point at which the loss stops descending or becomes ragged for `start_lr` and `end_lr` respectively. In the plot below, `start_lr = 0.0002` and `end_lr=0.2`.

**Installation**<P>

`pip install torch-lr-finder`

Install with the support of mixed precision training (see also this section):

`pip install torch-lr-finder -v --global-option="apex"`

**Implementation details and usage (Tweaked version from fastai)** <P>

Increases the learning rate in an exponential manner and computes the training loss for each learning rate. `lr_finder.plot()` plots the training loss versus logarithmic learning rate.

```python
from torch_lr_finder import LRFinder

model     = ...
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-7, weight_decay=1e-2)
lr_finder = LRFinder(model, optimizer, criterion, device="cuda")
lr_finder.range_test(trainloader, end_lr=100, num_iter=100)
lr_finder.plot()  # to inspect the loss-learning rate graph
lr_finder.reset() # to reset the model and optimizer to their initial state
```

**Leslie Smith's approach**<P>

Increases the learning rate linearly and computes the evaluation loss for each learning rate. `lr_finder.plot()` plots the evaluation loss versus learning rate. This approach typically produces more precise curves because the evaluation loss is more susceptible to divergence but it takes significantly longer to perform the test, especially if the evaluation dataset is large.

```python
from torch_lr_finder import LRFinder

model     = ...
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.1, weight_decay=1e-2)
lr_finder = LRFinder(model, optimizer, criterion, device="cuda")
lr_finder.range_test(trainloader, val_loader=val_loader, end_lr=1, num_iter=100, step_mode="linear")
lr_finder.plot(log_lr=False)
lr_finder.reset()
```


## Define the VQ-VAE model

### Residual layers for Encoder and Decoder

In [ ]:
class ResidualLayer(nn.Module):
    """
    A residual block to be used on Encoder and Decoder built of:

    -> + -> ReLU -> Conv2d -> ReLU -> Conv2d -> (+) ->
       |                                         |
       +-----------------------------------------+

    Arguments:
    * channels:  The number of input/hidden/output channels.
    """

    def __init__(self, channels):
        super(ResidualLayer, self).__init__()
        self.res_block = nn.Sequential(
            nn.ReLU(True),
            nn.Conv2d(
                in_channels  = channels,
                out_channels = channels,
                kernel_size  = 3,
                stride       = 1,
                padding      = 1,
                bias         = False,
            ),
            #nn.BatchNorm2d(channels),
            nn.ReLU(True),
            nn.Conv2d(
                in_channels  = channels,
                out_channels = channels,
                kernel_size  = 1,
                stride       = 1,
                bias         = False,
            ),
            #nn.BatchNorm2d(channels),
        )

    def forward(self, x):
        x = x + self.res_block(x)
        return x

# ----------------------------------------------------------------

class ResidualStack(nn.Module):
    """
    A stack of residual blocks to be used on Encoder and Decoder.

    Arguments:
    * channels:      The number of input/hidden/output channels
    * n_res_layers:  The number of residual blocks to stack.
    """

    def __init__(self, channels, n_res_layers):
        super(ResidualStack, self).__init__()
        self.n_res_layers = n_res_layers
        self.stack = nn.ModuleList(
            [ ResidualLayer(channels) ] * n_res_layers
        )

    def forward(self, x):
        for layer in self.stack:
            x = layer(x)
        return x

### Encoder module 

In [ ]:
class Encoder(nn.Module):
    """
    The encoder network that parameterizes the q_phi (z|x) model.
    Given a data sample x, q_phi maps it to the latent space x -> z_e.
    For a VQ VAE, q_phi outputs parameters of a categorical distribution.

    Arguments:
    * in_channels:    The number of input channels.
    * hidden_dim:     The encoder hidden size.
    * embedding_dim:  Embedding/latent vectors size.
    * n_res_layers:   The number of residual blocks to stack.
    * num_stride2:    Number of convolution layers with stride=2.
    """

    def __init__(self, in_channels, hidden_dim, embedding_dim, n_res_layers, num_stride2):
        super(Encoder, self).__init__()
        kernel = []
        stride = []
        assert num_stride2 <= 3, print(f'Hyperparameter "num_stride2" must be <= 3!')
        for n in range(3):
            if n < num_stride2:
                stride_n = 2
                kernel_n = 4
            else:
                stride_n = 1
                kernel_n = 3
            stride.append(stride_n)
            kernel.append(kernel_n)

        self.conv_stack = nn.Sequential(
            nn.Conv2d(
                in_channels  = in_channels,
                out_channels = hidden_dim, 
                kernel_size  = kernel[0],
                stride       = stride[0],
                padding      = 1,
            ),
            #nn.BatchNorm2d(hidden_dim),
            nn.ReLU(),
            nn.Conv2d(
                in_channels  = hidden_dim, 
                out_channels = hidden_dim,
                kernel_size  = kernel[1],
                stride       = stride[1],
                padding      = 1,
            ),
            #nn.BatchNorm2d(hidden_dim),
            nn.ReLU(),
            nn.Conv2d(
                in_channels  = hidden_dim,
                out_channels = embedding_dim,
                kernel_size  = kernel[2],
                stride       = stride[2],
                padding      = 1,
            ),
            #nn.BatchNorm2d(embedding_dim),
            ResidualStack(
                channels     = embedding_dim,
                n_res_layers = n_res_layers,
            )
        )

    def forward(self, x):
        return self.conv_stack(x)

### Vector Quantization module

In [ ]:
class VectorQuantizer(nn.Module):
    """
    VQ-VAE quantization module: given an input continuous tensor, returns the closest 
    discrete tensor from the embedding space (codebook).

    Arguments:
    * n_embeddings (int):  The number 'K' of vectors in the embedding space.
    * embedding_dim (int): The dimensionality 'D' of the tensors in the embedding 
                           space; inputs to the modules must be in this format as well.
    * beta (float):        Scalar that defines the weight of the commitment term
                           of the loss (variable 'beta' in equation 3 of VQ-VAE paper).
    """

    def __init__(self, n_embeddings, embedding_dim, beta):
        super(VectorQuantizer, self).__init__()
        self.n_embeddings  = n_embeddings
        self.embedding_dim = embedding_dim
        self.beta          = beta

        # nn.Embedding is a lookup table that stores a fixed dictionary (index:vector) with fixed size.
        # Embeddings are stored in variable 'weight' and retrieved using their indices.
        self.embeddings = nn.Embedding(self.n_embeddings, self.embedding_dim)

        # Initialize embeddings with uniform random numbers in range [-1/n_e : 1/n_e]
        self.embeddings.weight.data.uniform_(
            -1.0 / self.n_embeddings,
            1.0 / self.n_embeddings
        )

    # 'z' is the encoder output 'z_e' of the VQ-VAE
    def forward(self, z):
        """
        Inputs the output of the encoder network z and maps it to a discrete 
        one-hot vector that is the index of the closest embedding vector e_j

        z (continuous) -> z_q (discrete)

        z.shape = (batch, channel, height, width)

        quantization pipeline:

            1. get encoder input (B,C,H,W)
            2. flatten input to (B*H*W,C)

        """
        # z.shape = [BS, C, H, W] -> [BS, H, W, C]
        z = z.permute(0, 2, 3, 1).contiguous()

        # It is assumed that the number of channels (C) in the quantizer input
        # is the equal to the embedding dimension (D).
        #
        # [BS, H, W, C] -> [BS*H*W, C]
        z_flattened = z.view(-1, self.embedding_dim) # '-1' <=> adjust this dimension to the correct value

        # Obtain the index of the embedding that is closest to each z_flattened[i]
        encoding_indices = self.get_code_indices(z_flattened)

        # Get the closest embeddings given their indices
        z_quantized      = self.quantize(encoding_indices)

        # Reorganize the shape of the embeddings to be [BS, H, W, C]
        z_quantized        = z_quantized.view_as(z)

        # Quantization loss: to optimize the embedding space (e) by making it 
        # closer to the encoder output (z_e)
        # SUM ||sg[z_e] - e_k||^2_2
        quantization_loss = F.mse_loss(z_quantized, z.detach())

        # Commitment loss: to optimize the encoder and make its output (z_e)
        # closer to the embedding sapce (e)
        # SUM ||z_e - sg[e_k]||^2_2
        commit_loss = F.mse_loss(z, z_quantized.detach())

        # loss = quantization loss + beta * commitment loss
        loss = quantization_loss + self.beta * commit_loss

        # Preserve the gradients
        z_quantized = z + (z_quantized - z).detach()

        z_quantized = z_quantized.permute(0, 3, 1, 2).contiguous()

        # Calculate the perplexity = Exp ( −Mean { Log [p(e) + epsilon] } )

        embed_mean_probs = torch.bincount(encoding_indices)/encoding_indices.shape[0]
        embed_entropy    = embed_mean_probs * torch.log(embed_mean_probs + 1e-10)
        perplexity       = torch.exp(-torch.sum(embed_entropy))

        # print(f'ENC_INDEXES ({encoding_indices.shape}): {encoding_indices}')
        # print(f'MEAN PROBS  ({embed_mean_probs.shape}): {embed_mean_probs}')
        # print(f'ENTROPY     ({embed_entropy.shape}):    {embed_entropy}')
        # print(f'PERPLEXITY  ({perplexity.shape}):       {perplexity}')

        return z_quantized, loss, perplexity

    def get_code_indices(self, flat_z):
        '''
        Get the indices of the embedding vectors that are closest to 'flat_z'.
        '''

        # Calculate the L2 distance from each z[i] to every embedding[i].
        # 
        # L2 = (z-embedding)^2 = z^2 + embedding^2 - 2*z*embedding

        # print(f'FLAT_Z shape: {flat_z.shape}')
        # print(f'EMBED WEIGHTS shape: {self.embeddings.weight.shape}')
        # print(f'EMBED WEIGHTS transposed shape: {self.embeddings.weight.t().shape}')

        distances = (
            torch.sum(flat_z ** 2, dim=1, keepdim=True) +
            torch.sum(self.embeddings.weight ** 2, dim=1) -
            2. * torch.matmul(flat_z, self.embeddings.weight.t())
        ) # shape = [N, M]

        # Find the index of the embedding vector e_k that is closest to x[i] 
        encoding_indices = torch.argmin(distances, dim=1) # shape = [N,]

        return encoding_indices

    def quantize(self, encoding_indices):
        """
        Returns the embedding tensor given a batch of indices.
        """
        return self.embeddings(encoding_indices)

### Decoder module

In [ ]:
class Decoder(nn.Module):
    """
    The decoder network that parameterizes p_theta (x|z) model.
    Given a latent sample z, p_theta maps it back to the image space z -> x.

    Arguments:
    * in_dim:         The number of input channels.
    * out_channels:   The number of output channels.
    * hidden_dim:     The decoder hidden size.
    * n_res_layers:   The number of residual blocks to stack.
    * num_stride2:    Number of transpose convolution layers with stride=2.
    """

    def __init__(self, in_dim, out_channels, hidden_dim, n_res_layers, num_stride2):
        super(Decoder, self).__init__()
        kernel = []
        stride = []
        assert num_stride2 <= 3, print(f'Hyperparameter "num_stride2" must be <= 3!')
        for n in range(3):
            if n < num_stride2:
                stride_n = 2
                kernel_n = 4
            else:
                stride_n = 1
                kernel_n = 3
            stride.append(stride_n)
            kernel.append(kernel_n)

        self.inverse_conv_stack = nn.Sequential(
            ResidualStack(
                channels     = in_dim,
                n_res_layers = n_res_layers,
            ),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                in_channels  = in_dim,
                out_channels = hidden_dim,
                kernel_size  = kernel[0],
                stride       = stride[0],
                padding      = 1,
            ),
            # nn.BatchNorm2d(hidden_dim),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                in_channels  = hidden_dim,
                out_channels = hidden_dim,
                kernel_size  = kernel[1],
                stride       = stride[1],
                padding      = 1,
            ),
            # nn.BatchNorm2d(hidden_dim),
            nn.ReLU(True),
            nn.ConvTranspose2d(
                in_channels  = hidden_dim,
                out_channels = out_channels,
                kernel_size  = kernel[2],
                stride       = stride[2],
                padding      = 1,
            ),
        )

    def forward(self, x):
        return self.inverse_conv_stack(x)

### VQ-VAE module

In [ ]:
class VQVAE(nn.Module):

    def __init__(
        self,
        in_channels,
        out_channels,
        hidden_dim,
        n_res_layers,
        num_stride2,
        n_embeddings,
        embedding_dim,
        beta,
        save_img_embedding_map=False
        ):
        super(VQVAE, self).__init__()

        # encode image into continuous latent space
        self.encoder = Encoder(
            in_channels    = in_channels,
            hidden_dim     = hidden_dim,
            embedding_dim  = embedding_dim,
            n_res_layers   = n_res_layers,
            num_stride2    = num_stride2,
        )

        # pass continuous latent vector through discretization bottleneck
        self.vector_quantization = VectorQuantizer(
            n_embeddings  = n_embeddings,
            embedding_dim = embedding_dim,
            beta          = beta,
        )

        # decode the discrete latent representation
        self.decoder = Decoder(
            in_dim         = embedding_dim,
            out_channels   = out_channels,
            hidden_dim     = hidden_dim,
            n_res_layers   = n_res_layers,
             num_stride2    = num_stride2,
        )

        if save_img_embedding_map:
            self.img_to_embedding_map = {i: [] for i in range(n_embeddings)}
        else:
            self.img_to_embedding_map = None

    def forward(self, x, verbose=False):

        z_e = self.encoder(x)

        z_q, embedding_loss, perplexity = self.vector_quantization(z_e)
        x_pred = self.decoder(z_q)

        if verbose:
            print('original data shape:', x.shape)
            print('encoded data shape:',  z_e.shape)
            print('reconst data shape:',  x_pred.shape)
            assert False

        return x_pred, embedding_loss, perplexity

## Functions to save and load the model

In [ ]:
def save_model_and_results(model, results, hyperparameters, file_name):
    results_to_save = {
        'model': model.state_dict(),
        'results': results,
        'hyperparameters': hyperparameters,
    }

    torch.save(
        results_to_save,
        file_name,
    )

In [ ]:
def load_model(model, file_name, config, device):
    '''
    Given an instance of a model, loads from file 'file_name':
    (i)   the model weights,
    (ii)  the results obtained during the model training and
    (iii) the training hyperparameters used to train the model,
    and put the model on 'device'.

    Returns the loaded results and the loaded hyperparameters.
    '''

    results_loaded = torch.load(file_name)

    model.load_state_dict(results_loaded['model'])

    # Returns the saved results and the saved hyperparameters
    return results_loaded['results'], results_loaded['hyperparameters']

## Instantiate the VQ-VAE model

In [ ]:
# Instantiate the VQ-VAE model
vqvae_model = VQVAE(
    in_channels     = config["model_params"]["num_channels"],
    out_channels    = config["model_params"]["num_channels"],
    hidden_dim      = config["model_params"]["encoder_hidden_dim"],
    n_res_layers    = config["model_params"]["num_residual_layers"],
    num_stride2     = config["model_params"]["num_stride2"],
    n_embeddings    = config["model_params"]["embedding_number"],
    embedding_dim   = config["model_params"]["embedding_dim"],
    beta            = config["model_params"]["beta"],
).to(device)

# Define the optimizer
optimizer = torch.optim.Adam(
    vqvae_model.parameters(),
    lr      = config["exp_params"]["learning_rate"],
    amsgrad = True
)

# Define the learning rate scheduler

if config["exp_params"]["vqvae_scheduler"] == "RLRonPlateau":
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode      = 'min',
        factor    = 0.75, 
        patience  = 5,
        threshold = 2.5e-3,
    )
    print(f'[INFO] Selecting RLRonPlateau learning rate scheduler')
elif config["exp_params"]["vqvae_scheduler"] == "CyclicLR":
    scheduler = torch.optim.lr_scheduler.CyclicLR(
        optimizer,
        base_lr      = config["exp_params"]["learning_rate"], # Initial and minimum learning rate
        max_lr       = 0.1,          # Maximum learning rate
        step_size_up = 5,            # Number of training iterations in the increasing half cycle
        mode         = "triangular", # Can be "triangular", "triangular2", "exp_range"
    )
    print(f'[INFO] Selecting CyclicLR learning rate scheduler')
elif config["exp_params"]["vqvae_scheduler"] == "CosineAnnealingWarmRestarts":
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, 
        T_0        = 10,    # Number of iterations in the first restart (cycle), which defines
                            # the number of iterations since LR_max=lr until LR_min=eta_min.
        T_mult     = 1,     # A factor that multiplies T_0​ after each restart (cycle), which 
                            # specifies the number of iterations in next cycle (default=1).
        eta_min    = 1e-5,  # The minimum learning rate.
        last_epoch = -1,    # When last_epoch=-1, sets initial learning rate as the 'lr' 
                            # value defined in 'optimizer'.
    )
    print(f'[INFO] Selecting CosineAnnealingWarmRestarts learning rate scheduler')
else:
    scheduler = None


## Load the VQ-VAE model from file

In [ ]:
file_saved_model  = f'models/{config["exp_params"]["vqvae_save_file"]}.pth'

if LOAD_VQVAE_MODEL == True:
    _, _ = load_model(vqvae_model, file_saved_model, config, device)

## Train the VQ-VAE model

In [ ]:
def train_step_vqvae(
        model:          torch.nn.Module,
        dataloader:     torch.utils.data.DataLoader,
        optimizer:      torch.optim.Optimizer,
        scheduler:      torch.optim.lr_scheduler,
        train_variance: torch.Tensor,
        epoch:          int,
		log_interval:   int,
        results:        Dict[str, List[float]],
        device:         torch.device,
    ) -> Tuple[float, float, float]:
    """
	Trains the VQ-VAE 'model' for a single epoch.

	Turns the model into the training mode and then
	runs through all of the required training steps (forward
	pass, loss calculation, optimizer step).

	Arguments:
		model:          A VQ-VAE model to be trained.
		dataloader:     A DataLoader instance for the model to be trained on.
		optimizer:      A PyTorch optimizer to help minimize the loss function.
        scheduler:      Learning rate scheduler to apply.
		train_variance: The training dataset variance.
		epoch:          The current training epoch number.
  		log_interval:   Interval (in batches) between successive logs.
		results:        Dictionary to append the training results.
		device:         A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A tuple of loss, reconstruction loss and perplexity metrics.
        In the form (loss, reconst_loss, perplexity).
    """
    train_loss         = 0
    train_reconst_loss = 0
    train_perplexity   = 0

    # Put the model in train mode
    model.train()

    for i, x in enumerate(tqdm(dataloader, desc=f'Epoch {epoch+1}')):

        # 1. Send data to target device
        x = x.to(device)

        # 2. Reset the gradients of the loss
        optimizer.zero_grad()

        # 3. Forward pass through the model
        x_pred, embedding_loss, perplexity = model(x)

        # 4. Calculate the loss
        reconst_loss = torch.mean((x_pred - x)**2) / train_variance
        loss         = reconst_loss + embedding_loss

        # 5. Calculate the gradient of the loss relative to all model parameters
        loss.backward()

        # 6. Update parameters using the calculated gradients
        optimizer.step()

        # 7. Accumulate metrics
        train_loss         += loss.item()
        train_reconst_loss += reconst_loss.item()
        train_perplexity   += perplexity.item()

        # 8. Save results
        results["train_reconst_loss"].append(reconst_loss.item())
        results["train_perplexity"].append(perplexity.item())
        results["train_loss"].append(loss.item())
        results["train_n_updates"] = i

        # 9. Print progress information
        if i % log_interval == 0:

            print(
                'Update #', i,
                'Reconst loss: ', np.mean(results["train_reconst_loss"][-log_interval:]),
                'Loss: ',         np.mean(results["train_loss"][-log_interval:]),
                'Perplexity: ',   np.mean(results["train_perplexity"][-log_interval:])
            )

    # Compute average metrics across all batches
    train_loss         = train_loss / len(dataloader)
    train_reconst_loss = train_reconst_loss / len(dataloader)
    train_perplexity   = train_perplexity / len(dataloader)

    return train_loss, train_reconst_loss, train_perplexity

In [ ]:
def val_step_vqvae(
    model:          torch.nn.Module,
    dataloader:     torch.utils.data.DataLoader,
    train_variance: torch.Tensor,
    epoch:          int,
    log_interval:   int,
    results:        Dict[str, List[float]],
    device:         torch.device,
    ) -> Tuple[float, float, float]:
    """
    Evaluate a VQ-VAE model for a single epoch.

    Turns a VQ-VAE 'model' to evaluation mode and then performs
    a forward pass on a validation dataset.

    Arguments:
        model:          A VQ-VAE model to be evaluated.
        dataloader:     A DataLoader instance for the model to be evaluated on.
        train_variance: The training dataset variance.
        epoch:          The current validation epoch number.
        log_interval:   Interval (in batches) between successive logs.
        results:        Dictionary to append the validation results.
        device:         A target device to compute on (e.g. 'cuda' or 'cpu').

    Returns:
        A tuple of loss, reconstruction loss and perplexity.
        In the form (loss, reconst_loss, perplexity).
    """
    # Put the model in evaluation mode
    model.eval()

    # Initialize the validation metrics
    val_loss         = 0
    val_reconst_loss = 0
    val_perplexity   = 0

    # Turn on inference context manager
    with torch.inference_mode():

        # Loop through the DataLoader batches
        for i, x in enumerate(tqdm(dataloader, desc=f'Epoch {epoch+1}')):

            # 1. Send data to target device
            x = x.to(device)

            # 2. Forward pass
            x_pred, embedding_loss, perplexity = model(x)

            # 3. Calculate the loss
            reconst_loss = torch.mean((x_pred - x)**2) / train_variance
            loss         = reconst_loss + embedding_loss

            # 4. Accumulate the metrics
            val_loss         += loss.item()
            val_reconst_loss += reconst_loss.item()
            val_perplexity   += perplexity.item()

            # 5. Save the partial results
            results["val_reconst_loss"].append(reconst_loss.item())
            results["val_perplexity"].append(perplexity.item())
            results["val_loss"].append(loss.item())
            results["val_n_updates"] = i

            # 6. Print progress information
            if i % log_interval == 0:

                print(
                    'Update #', i,
                    'Val reconst loss: ', np.mean(results["val_reconst_loss"][-log_interval:]),
                    'Val loss: ',         np.mean(results["val_loss"][-log_interval:]),
                    'Val perplexity: ',   np.mean(results["val_perplexity"][-log_interval:])
                )

    # Compute average metrics across all batches
    val_loss         = val_loss / len(dataloader)
    val_reconst_loss = val_reconst_loss / len(dataloader)
    val_perplexity   = val_perplexity / len(dataloader)

    # Save a sample of 8 images and recontructions to file at the end of each validation epoch
    sample     = x[:sample_size]
    sample_out = x_pred[:sample_size]

    utils.save_image(
        torch.cat([sample, sample_out], 0),
        f'results/reconstruct_{str(epoch + 1).zfill(5)}_{str(i).zfill(5)}.png',
        nrow        = sample_size,
        normalize   = False,
        value_range = (-1, 1),
    )

    return val_loss, val_reconst_loss, val_perplexity

In [ ]:
def train_vqvae(
        model:            torch.nn.Module,
        train_dataloader: torch.utils.data.DataLoader,
        val_dataloader:   torch.utils.data.DataLoader,
        optimizer:        torch.optim.Optimizer,
        scheduler:        torch.optim.lr_scheduler,
        config:           Dict,
        train_variance:   torch.Tensor,
        device:           torch.device,
    ) -> Dict[str, List]:
    """
    Trains and validates a VQ-VAE model.

    Passes a target PyTorch models through train_step_vqvae() and val_step_vqvae()
    functions for a number of epochs, training and validating the model
    in the same epoch loop.

    Calculates, prints and stores evaluation metrics throughout.

    Arguments:
        model:            A VQ-VAE model to be trained and validated.
        train_dataloader: A DataLoader instance for the model to be trained on.
        test_dataloader:  A DataLoader instance for the model to be tested on.
        optimizer:        A PyTorch optimizer to help minimize the loss function.
        scheduler:        Learning rate scheduler to apply.
        config:           A Dictionary containing the training configuration.
        train_variance:   The training dataset variance.
        device:           A target device to compute on (e.g. "cuda" or "cpu").

    Returns:
        A dictionary of training and validation results.
        Each metric has a list of values, one per batch.
        In the form:
            {
            train_reconst_loss: [...],
            train_loss:         [...],
            train_perplexity:   [...],
            val_reconst_loss:   [...],
            val_loss:           [...],
            val_perplexity:     [...],
            }
    """

    log_interval    = config["exp_params"]["log_interval"]
    train_variance  = torch.mean(train_variance)
    file_save_model = f'models/{config["exp_params"]["vqvae_save_file"]}.pth'
    n_train_results = log_interval * len(train_dataloader)
    #n_val_results  = log_interval * len(val_dataloader)

    # Create an empty dictionary of results
    results = {
        'train_n_updates':    0,
        'train_reconst_loss': [],
        'train_loss':         [],
        'train_perplexity':   [],
        'val_n_updates':      0,
        'val_reconst_loss':   [],
        'val_loss':           [],
        'val_perplexity':     [],
    }

    for epoch in trange(config["exp_params"]["num_epochs"]):

        train_loss, train_reconst_loss, train_perplexity = train_step_vqvae(
            model          = model,
            dataloader     = train_dataloader,
            optimizer      = optimizer,
            scheduler      = scheduler,
            train_variance = train_variance,
            epoch          = epoch,
            log_interval   = log_interval,
            results        = results,
            device         = device,
       )

        val_loss, val_reconst_loss, val_perplexity = val_step_vqvae(
            model          = model,
            dataloader     = val_dataloader,
            train_variance = train_variance,
            epoch          = epoch,
            log_interval   = log_interval,
            results        = results,
            device         = device,
        )

        try:
            # Log metrics to W&B
            wandb.log(
                {
                "train_vqvae_reconst_loss": np.mean(results["train_reconst_loss"][-n_train_results:]),
                "train_vqvae_loss":         np.mean(results["train_loss"][-n_train_results:]),
                "train_vqvae_perplexity":   np.mean(results["train_perplexity"][-n_train_results:]),
                "val_vqvae_reconst_loss":   np.mean(results["val_reconst_loss"][-n_train_results:]),
                "val_vqvae_loss":           np.mean(results["val_loss"][-n_train_results:]),
                "val_vqvae_perplexity":     np.mean(results["val_perplexity"][-n_train_results:]),
                "vqvae_epoch":              epoch+1,
                }
            )
        except Exception as ex:
            print(f'An exception of type {type(ex).__name__} occurred. Arguments:\n{ex.args!r}')

        # Save the model, the results, and the hyperparameters
        if config["exp_params"]["save_model"] == True:

            hyperparameters = config
            save_model_and_results(
                model,
                results,
                hyperparameters,
                file_save_model,
            )

        # Update the learning rate if necessary
        if config["exp_params"]["vqvae_scheduler"] == "RLRonPlateau":
            old_lr = optimizer.param_groups[0]['lr']
            scheduler.step(val_loss)
            new_lr = optimizer.param_groups[0]['lr']
            if new_lr != old_lr:
                print(f'[INFO] Learning rate was altered from {old_lr} to {new_lr}.')
        elif config["exp_params"]["vqvae_scheduler"] == "CyclicLR":
            scheduler.step()
        elif config["exp_params"]["vqvae_scheduler"] == "CosineAnnealingWarmRestarts":
            scheduler.step(epoch)

    # Return the filled results at the end of the epochs
    return results

In [ ]:
if SKIP_TRAIN_VQVAE_MODEL == False:

    results = train_vqvae(
        vqvae_model,
        train_dataloader,
        val_dataloader,
        optimizer,
        scheduler,
        config,
        train_variance,
        device
    )

## Plot the training loss, reconstruction loss and perplexity

In [ ]:
if SKIP_TRAIN_VQVAE_MODEL == False:

    results_df = pd.concat({k: pd.Series(v) for k, v in results.items()}, axis=1)

    results_df.to_csv(
        f"results/{BASE_FILE_NAME}_results.csv",
        sep=',',
        index=False,
        encoding='utf-8'
    )

In [ ]:
if SKIP_TRAIN_VQVAE_MODEL == False:
    results_df.head()

In [ ]:
if SKIP_TRAIN_VQVAE_MODEL == False:
    results_df["train_loss"].plot()

In [ ]:
if SKIP_TRAIN_VQVAE_MODEL == False:
    results_df["val_loss"].plot()

In [ ]:
if SKIP_TRAIN_VQVAE_MODEL == False:
    results_df["train_reconst_loss"].plot()

In [ ]:
if SKIP_TRAIN_VQVAE_MODEL == False:
    results_df["val_reconst_loss"].plot()

In [ ]:
if SKIP_TRAIN_VQVAE_MODEL == False:
    results_df["train_perplexity"].plot()

In [ ]:
if SKIP_TRAIN_VQVAE_MODEL == False:
    results_df["val_perplexity"].plot()

## Select 32 images and reconstruct them

In [ ]:
# Reconstruct images

# Get a batch of images from the validation set
test_loader_iter = iter(test_dataloader)
images           = next(test_loader_iter)

n_samples = 32
images    = images[:n_samples]

# Place the first 32 images on the 'device'

images = images.to(device)

# Put the model in evaluation mode
vqvae_model.eval()

with torch.inference_mode():

    # Pass the 32 validation images through the model
    reconst_images, e_loss, perplex = vqvae_model(images)

images         = images.permute(0, 2, 3, 1)         # BS,C,H,W --> BS,H,W,C
reconst_images = reconst_images.permute(0, 2, 3, 1) # BS,C,H,W --> BS,H,W,C

## Make a single image containing a 8x8 grid of images and upload it to W&B

The grid conatians a row of original images alternated with a row of reconstructed images.

In [ ]:
IMAGES_PER_ROW   = 8
HALF_NUMBER_ROWS = 4

images_p = images*255
images_p = images_p.cpu()

reconst_images_p = torch.clamp(reconst_images, min=0.0, max=1.0)
reconst_images_p = (reconst_images_p*255)
reconst_images_p = reconst_images_p.cpu()

bs, h, w, c = images_p.shape
bs *= 2
mixed_images = torch.zeros([bs, h, w, c])
for r in range(HALF_NUMBER_ROWS):
    mixed_images[r*2*IMAGES_PER_ROW:(r*2+1)*IMAGES_PER_ROW,:,:,:]  = images_p[r*IMAGES_PER_ROW:(r+1)*IMAGES_PER_ROW,:,:,:]
    mixed_images[(r*2+1)*IMAGES_PER_ROW:(r+1)*2*IMAGES_PER_ROW,:,:,:] = reconst_images_p[r*IMAGES_PER_ROW:(r+1)*IMAGES_PER_ROW,:,:,:]

mixed_images  = mixed_images.permute(0, 3, 1, 2)
grid        = make_grid(mixed_images, nrow=8, padding=1, pad_value=1)
grid        = grid.permute(1, 2, 0)
grid        = grid .numpy()

wandb_image = wandb.Image(grid, caption="VQ-VAE original vs reconstructed images")
wandb.log({"VQ-VAE original vs reconstructed images": wandb_image})

## Plot the 32 pairs of input and reconstructed images

In [ ]:
reconst_images  = np.array(
    np.clip(
        reconst_images.cpu().numpy(), 0., 1.) * 255,
        dtype=np.uint8
)
original_images = np.array(images.cpu().numpy() * 255, dtype=np.uint8)

reconst_images  = reconst_images.reshape(
    4,
    8,
    config["model_params"]["patch_size"],
    config["model_params"]["patch_size"],
    config["model_params"]["num_channels"]
)
original_images = original_images.reshape(
    4,
    8,
    config["model_params"]["patch_size"],
    config["model_params"]["patch_size"],
    config["model_params"]["num_channels"]
)

# Plot 4 times:
#  - a row with 8 input images         (on top)
#  - a row with 8 reconstructed images (below)

fig = plt.figure(figsize=(10, 10), constrained_layout=True)
gs  = fig.add_gridspec(8, 8)
for n_row in range(4):
    for n_col in range(8):
        f_ax = fig.add_subplot(gs[n_row * 2, n_col])
        img = original_images[n_row, n_col]
        f_ax.imshow(img)
        f_ax.axis("off")
        f_ax = fig.add_subplot(gs[n_row * 2 + 1, n_col])
        img  = reconst_images[n_row, n_col]
        f_ax.imshow(img)
        f_ax.axis("off")

## Train the Autoregressive Prior - PixelCNN

### Collect the indices of the closest embedding vector for all training set or load them from file

In [ ]:
import pickle

file_codebook = f'models/{config["exp_params"]["vqvae_save_file"]}.pkl'
train_indices = []

# Load the codebook indices from Pickle file ----------------------------
if SKIP_COMPUTE_CODEBOOK == True:

    with open(file_codebook, 'rb') as f:
        train_indices = pickle.load(f)

# Compute the codebook indices -----------------------------------------
else:

    # Put the VQ-VAE model in evaluation mode
    vqvae_model.eval()

    for images in train_dataloader:

        # Normalize images and put them in the 'device'
        # images = images - 0.5 # normalize to [-0.5, 0.5] # ########## COMMENTED
        images = images.to(device)

        with torch.inference_mode():

            # Pass a batch of images through the encoder
            ze = vqvae_model.encoder(images) # [BS, D, H, W]

            # Get dimensions of the batch of latent variables
            b, d, h, w = ze.size()

            # Make channels (embedding D) dimension the last one: [BS, D, H, W] -> [BS, H, W, D]
            ze = ze.permute(0, 2, 3, 1).contiguous()

            # Reshape the batch of latents: [BS, H, W, D] -> [BS*H*W, D]
            flat_ze = ze.reshape(-1, d)

            # Get the indices of the embedding vectors that are closest to 'flat_ze'
            encoding_indices = vqvae_model.vector_quantization.get_code_indices(flat_ze) # [BS*H*W,]

            # Reshape the indices: [BS*H*W] -> [BS,H,W]
            encoding_indices = encoding_indices.reshape(b, h, w)

            # Aggregate the indices of the closest embedding vector for all training set
            train_indices.append(encoding_indices.cpu())

    # Save the indices to a pickle file ---------------------------------------
    with open(file_codebook, 'wb') as f:
        pickle.dump(train_indices, f, protocol=pickle.HIGHEST_PROTOCOL)
        f.close()

### Masked Convolutions

In [ ]:
class MaskedConv2d(nn.Conv2d):
    """
    Implements a Conv2d with a mask applied to its weights.
    It is implemented by extending the 'Conv2d' class of 'pytorch.nn'.

    Arguments:
        * mask (torch.Tensor):        The mask tensor.
        * in_channels (int):          Number of channels in the input image.
        * out_channels (int):         Number of channels produced by the convolution.
        * kernel_size (int or tuple): Size of the convolving kernel.
    """

    def __init__(self, mask, in_channels, out_channels, kernel_size, **kwargs):

        super().__init__(in_channels, out_channels, kernel_size, **kwargs)

        # use nn.Module.register_buffer to register the mask persistently;
        # a buffer is part of model state but it is not parameters
        self.register_buffer('mask', mask[None, None])

    def forward(self, x):
        # Mask the Conv2d weights <=> multiply the Conv2d weights by the mask
        self.weight.data *= self.mask 
        # Pass input 'x' through the Conv2d with the masked weights
        return super().forward(x)

### Masked convolutions for the vertical stack of PixelCNN

In [ ]:
class VerticalStackConv(MaskedConv2d):
    '''
    Masked Conv2d for the vertical stack. The kernel size is n x n.

    If the mask is type "A", the filter central row and all rows below are set to zero.
    If the mask is type "B", the rows below the central row of the filter are set to zero.
    '''

    def __init__(self, mask_type, in_channels, out_channels, kernel_size, **kwargs):

        # Mask out all pixels that are below the actual row and also the actual row 
        # if mask type is "A".
        # If we want a more efficient implementation, we could reduce the height of 
        # the kernel to half value (k/2, k), but to keep the implementation simple 
        # we opt for using the mask.
        self.mask_type = mask_type # Type "A" or "B"

        # check if kernel size is a scalar 'int' value
        if isinstance(kernel_size, int):
            kernel_size = (kernel_size, kernel_size) # n x n kernel

        # Initialize the mask with zeros
        mask = torch.zeros(kernel_size)

        # Set the upper floor(kernel_size/2) rows of the mask to '1'
        mask[:kernel_size[0]//2, :] = 1.0

        # Type "A" mask masks: actual row and all rows below the actual row.
        #
        # Type "B" mask masks: all rows below the actual row.

        if self.mask_type == "B":
            mask[kernel_size[0]//2, :] = 1.0

        super().__init__(mask, in_channels, out_channels, kernel_size, **kwargs)

### Masked convolutions for the horizontal stack of PixelCNN

In [ ]:
class HorizontalStackConv(MaskedConv2d):
    '''
    Masked Conv2d for the horizontal stack. The kernel size is 1 x n.

    If the mask is type "A", the filter central element and all elements 
    to the right are set to zero.
    If the mask is type "B", all elements to the right of the central element 
    of the filter are set to zero.
    '''

    def __init__(self, mask_type, in_channels, out_channels, kernel_size, **kwargs):

        # Mask out all pixels on the left. Note that the kernel has
        # height=1 because we only look at the pixel in the same row.
        self.mask_type = mask_type # Type "A" or "B"

        # check if kernel size is a scalar 'int' value
        if isinstance(kernel_size, int):
            kernel_size = (1, kernel_size) # 1 x n kernel

        # Ensure that the first dimension of the kernel is '1'
        assert kernel_size[0] == 1

        if "padding" in kwargs:
            if isinstance(kwargs["padding"], int):
                kwargs["padding"] = (0, kwargs["padding"])

        # Initialize the mask with zeros
        mask = torch.zeros(kernel_size)

        # Set the left floor(kernel_size/2) elements of the mask to '1'
        mask[:, :kernel_size[1]//2] = 1.0

        # Type "A" mask masks: central element and all elements to the right.
        #
        # Type "B" mask masks: all elements to the right of the central element.

        if self.mask_type == "B":
            mask[:, kernel_size[1]//2] = 1.0

        super().__init__(mask, in_channels, out_channels, kernel_size, **kwargs)

### Single layer of the Gated PixelCNN

![](../fig/PixelCNN_gated_unit_1.png)

In [ ]:
class GatedMaskedConv(nn.Module):
    """
    Single layer of the Gated PixelCNN.
    It implements the computation graph shown in figure above.
    """

    def __init__(self, in_channels, kernel_size=3, dilation=1):
        super().__init__()

        padding = dilation * (kernel_size - 1) // 2

        # n x n convolution for the vertical stack
        self.conv_vert  = VerticalStackConv(
            mask_type   = "B",
            in_channels  = in_channels, 
            out_channels = 2*in_channels, 
            kernel_size  = kernel_size,
            padding      = padding,
            dilation     = dilation
        )

        # 1 x n convolution for the horizontal stack
        self.conv_horiz = HorizontalStackConv(
            mask_type    = "B",
            in_channels  = in_channels,
            out_channels = 2*in_channels,
            kernel_size  = kernel_size,
            padding      = padding,
            dilation     = dilation
        )

        # 1 x 1 convolution to feed features maps from vertical stack
        # into the horizontal stack
        self.conv_vert_to_horiz = nn.Conv2d(
            in_channels  = 2*in_channels,
            out_channels = 2*in_channels,
            kernel_size  = 1
        )

        # 1 x 1 convolution applied the output of the Gated Activation Unit 
        # in the horizontal stack
        self.conv_horiz_1x1 = nn.Conv2d(
            in_channels,
            in_channels, 
            kernel_size = 1
        )

    def forward(self, v_stack_in, h_stack_in):

        # ..............................................
        # Vertical stack (on the left of figure above)

        # n x n convolution applied to vertical stack input
        v_stack_features = self.conv_vert(v_stack_in)

        # Split the feature maps from n x n conv into 2 parts: 
        # one is for the tanh and the other is for the sigmoid
        v_activation, v_forget = v_stack_features.chunk(2, dim=1)

        # Multiply the outputs of tanh and sigmoid to produce the 
        # vertical stack output
        v_stack_out   = torch.tanh(v_activation) * torch.sigmoid(v_forget)

        # ...............................................
        # Horizontal stack (on the right of figure above)

        # 1 x n convolution applied to horizontal stack input
        h_stack_features = self.conv_horiz(h_stack_in)

        # Add 1 x n conv output with the vertical->horizontal 1 x 1 conv output 
        h_stack_features = h_stack_features + self.conv_vert_to_horiz(v_stack_features)

        # Split the feature maps from previous addition into 2 parts: 
        # one is for the tanh and the other is for the sigmoid
        h_activation, h_forget = h_stack_features.chunk(2, dim=1)

        # Multiply the outputs of tanh and sigmoid to produce new
        # horizontal feature maps
        h_stack_features = torch.tanh(h_activation) * torch.sigmoid(h_forget)

        # 1 x 1 convolution applied to the feature maps generated in previous step
        h_stack_out   = self.conv_horiz_1x1(h_stack_features)

        # Residual connection: adds the horizontal stack input with the 
        #                      output of the previous 1 x 1 conv
        h_stack_out   = h_stack_out + h_stack_in

        return v_stack_out, h_stack_out

### Gated PixelCNN

In [ ]:
class GatedPixelCNN(nn.Module):
    '''
    The Gated PixelCNN module.
    It includes 'n_layers+2' layers:
    * a first layer including only 2 masked convolutions (of type "A").
    * 'n_layers' gated layers.
    * a final classification layer built with an ELU activation and a 1x1 Conv2d.
    '''

    def __init__(self, in_channels, hidden_channels, out_channels, n_layers):
        super().__init__()

        # Initial conv layer uses a type "A" mask
        self.conv_vstack = VerticalStackConv(
            mask_type   = "A", 
            in_channels  = in_channels, 
            out_channels = hidden_channels, 
            kernel_size  = 3, 
            padding      = 1,
        )
        self.conv_hstack = HorizontalStackConv(
            mask_type   = "A", 
            in_channels  = in_channels, 
            out_channels = hidden_channels, 
            kernel_size  = 3, 
            padding      = 1,
        )

        # Convolution block of PixelCNN. It uses dilation instead of downscaling,
        # as occurs in the encoder-decoder architecture of PixelCNN++.
        self.conv_layers = nn.ModuleList()

        for l in range(n_layers):
            self.conv_layers.append(
                GatedMaskedConv(hidden_channels)
            )

        # PixelCNN output classification 1 x 1 convolution
        self.conv_out = nn.Conv2d(hidden_channels, out_channels, kernel_size=1)

    def forward(self, x):

        # First PixelCNN layer: it includes only 2 masked convolutions (of type "A")
        v_stack = self.conv_vstack(x)
        h_stack = self.conv_hstack(x)

        # Create 'n_layers' gated layers
        for layer in self.conv_layers:
            v_stack, h_stack = layer(v_stack, h_stack)

        # Output 1 x 1 classification convolution.
        # It includes an ELU activation, which is applied to the output of horizontal stack
        # of the last Gated layer, to introduce non-linearity on the last residual connection.
        out = self.conv_out(F.elu(h_stack))

        return out

### Instantiate the Gated PixelCNN model

In [ ]:
# Instantiate the PixelCNN model

pixelcnn  = GatedPixelCNN(
    in_channels     = config["model_params"]["embedding_number"],
    hidden_channels = config["model_params"]["pixelcnn_hidden_channels"],
    out_channels    = config["model_params"]["embedding_number"],
    n_layers        = config["model_params"]["gated_layers"],
)

# Put the model in the 'device'

pixelcnn  = pixelcnn.to(device)

# Select the optimizer

optimizer = torch.optim.Adam(
    pixelcnn.parameters(),
    lr = config["exp_params"]["pixelcnn_learning_rate"]
)

# Define the learning rate scheduler

if config["exp_params"]["pixelcnn_scheduler"] == "RLRonPlateau":
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode      = 'min',
        factor    = 0.9, 
        patience  = 5,
        threshold = 2.5e-3,
    )
elif config["exp_params"]["pixelcnn_scheduler"] == "CyclicLR":
    scheduler = torch.optim.lr_scheduler.CyclicLR(
        optimizer,
        base_lr      = config["exp_params"]["learning_rate"], # Initial and minimum learning rate
        max_lr       = 0.1,          # Maximum learning rate
        step_size_up = 5000,         # Number of training iterations in the increasing half cycle
        mode         = "triangular", # Can be "triangular" or "triangular2" or "exp_range"
    )
elif config["exp_params"]["pixelcnn_scheduler"] == "CosineAnnealingWarmRestarts":
    scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
        optimizer, 
        T_0        = 10000, # Number of iterations in the first restart (cycle), which defines the number of 
                            # iterations since LR_max=lr until LR_min=eta_min.
        T_mult     = 1,     # A factor that multiplies T_0​ after each restart (cycle), which specifies the
                            # number of iterations in next cycle (default=1).
        eta_min    = 1e-6,  # The minimum learning rate
        last_epoch = -1,    # When last_epoch=-1, sets initial learning rate as the 'lr' value defined on 'optimizer' 
    )
else:
    scheduler = None


### Load the Gated PixelCNN model from file

In [ ]:
file_saved_model  = f'models/{config["exp_params"]["pixelcnn_save_file"]}.pth'

if LOAD_PIXELCNN_MODEL == True:
    _, _ = load_model(pixelcnn, file_saved_model, config, device)

### Train the Gated PixelCNN model

In [ ]:
def train_pixelcnn(
        model:            torch.nn.Module,
        train_indices:    List,
        optimizer:        torch.optim.Optimizer,
        scheduler:        torch.optim.lr_scheduler,
        config:           Dict,
        device:           torch.device,
    ) -> Dict[str, List]:
    '''
    Train the Gated PixelCNN model.

    Arguments:
        model:         A PixelCNN model to be trained.
        train_indices: List with the indices of the closest
                       embedding vector for all training set.
        optimizer:     A PyTorch optimizer to help minimize the loss function.
        scheduler:     The learning rate scheduler to apply.
        config:        A Dictionary containing the training configuration.
        device:        A target device to compute on (e.g. 'cuda' or 'cpu').

    Returns:
        The dictionary containing a list with the loss values.
    '''
    log_interval     = config["exp_params"]["log_interval"]
    epochs           = config["exp_params"]["pixelcnn_num_epochs"]
    num_embeddings   = config["model_params"]["embedding_number"]
    file_save_model  = f'models/{config["exp_params"]["pixelcnn_save_file"]}.pth'
    old_lr           = config["exp_params"]["pixelcnn_learning_rate"]

    pixelcnn_results = {
        'loss': [],
    }

    model.train()

    # Training loop

    for epoch in trange(epochs):

        train_loss = 0

        # Iterate through the indices of the embeddings closest to the latent vectors z_e,
        # for all the training set

        for i, indices in enumerate(tqdm(train_indices, desc=f'Epoch {epoch+1}')):

            start_time = time.time()

            # 1. Put the indices in the 'device'
            indices         = indices.to(device)

            # 2. One-hot encode each index in a vector with size = num_embeddings
            one_hot_indices = F.one_hot(indices, num_embeddings).float().permute(0, 3, 1, 2).contiguous()

            # 3. Pass the one-hot encoded indices through the PixelCNN
            outputs         = model(one_hot_indices)

            # 4. Calculate the cross-entropy loss between input indices and predicted indices
            loss            = F.cross_entropy(outputs, indices)

            # 5. Reset the loss gradients
            optimizer.zero_grad()

            # 6. Calculate the gradient of the loss relative to the model parameters
            loss.backward()

            # 7. Update model parameters using the calculated gradients
            optimizer.step()

            # 8. Accumulate and save the loss
            train_loss += loss.item()
            pixelcnn_results["loss"].append(loss.item())

            # 9. Update the learning rate if necessary
            if config["exp_params"]["pixelcnn_scheduler"] == "RLRonPlateau":
                scheduler.step(train_loss)
                new_lr = optimizer.param_groups[0]['lr']
                if new_lr != old_lr:
                    print(f'[INFO] Learning rate was altered from {old_lr} to {new_lr}.')
                    old_lr = new_lr
            elif config["exp_params"]["pixelcnn_scheduler"] == "CyclicLR":
                scheduler.step()
            elif config["exp_params"]["pixelcnn_scheduler"] == "CosineAnnealingWarmRestarts":
                scheduler.step(epoch + i / train_data_len)

            # 10. Display progress information
            if (i + 1) % log_interval == 0:

                end_time = time.time()

                print(
                    f'\tIteration: [{i+1}/{len(train_indices)} \
                    ({((i+1)*100) / len(train_indices) :.0f}%)] \
                    \tLoss: {np.asarray(pixelcnn_results["loss"])[-log_interval:].mean(0)} \
                    Time: {end_time - start_time}'
                )

            # 11. Log the metrics to W&B
            wandb.log(
                {
                "pixelcnn_train_loss": loss.item(),
                "pixelcnn_epoch":      epoch+1,
                "pixelcnn_batch":      i+1,
                }
            )

        # Save the model, the results, and the hyperparameters
        if config["exp_params"]["pixelcnn_save_model"] == True:

            hyperparameters = config

            save_model_and_results(
                model,
                pixelcnn_results,
                hyperparameters,
                file_save_model,
            )

        # Compute average loss across all batches
        train_loss  = train_loss / len(train_indices)

    return pixelcnn_results


In [ ]:
# Train the Gated PixelCNN

if SKIP_TRAIN_PIXELCNN_MODEL == False:

    pixelcnn_results = train_pixelcnn(
        pixelcnn,
        train_indices,
        optimizer,
        scheduler,
        config,
        device,
    )

## Use the Gated PixelCNN to generate new images

In [ ]:
N_RUNS = 16

for run in tqdm(range(N_RUNS)):

    # ###################################################################
    # 1. Use the Gated PixelCNN to generate priors

    # Create an empty array of priors
    H_W_PRIORS = int(config["model_params"]["patch_size"] / (2 ** config["model_params"]["num_stride2"]))

    n_samples  = 64
    prior_size = (H_W_PRIORS, H_W_PRIORS) # [h, w]
    priors     = torch.zeros((n_samples,) + prior_size, dtype=torch.long).to(device) #  [64, H_W_PRIORS, H_W_PRIORS]

    # Use the PixelCNN to generate priors ...........................

    # Put the model in evaluation mode
    pixelcnn.eval()

    # Iterate over the H_W_PRIORS x H_W_PRIORS priors element-by-element, since the generation has
    # to be done sequentially.

    for row in range(prior_size[0]):

        for col in range(prior_size[1]):

            with torch.inference_mode():
                # Feed the whole array and retrieve the probabilities
                # associated to each possible value of the actual element.
                one_hot_priors = F.one_hot(
                    priors,
                    config["model_params"]["embedding_number"]
                    ).float().permute(0, 3, 1, 2).contiguous() # [64, EMBED_DIM, H_W_PRIORS, H_W_PRIORS]
                logits = pixelcnn(one_hot_priors)

                # Get the probabilities
                probs = F.softmax(logits[:, :, row, col], dim=-1) # [64, EMBED_DIM]

                # Use the multinomial distribution, parameterized by the probabilities generated for
                # the actual element, to draw the value of the actual element and save
                # the values in the 'priors' tensor.
                priors[:, row, col] = torch.multinomial(probs, num_samples=1).squeeze(dim=-1) # [64]

    # ###################################################################
    # 2. Generate new images with VQ-VAE decoder using as input the embeddings
    #    that correspond to the generated prior indices

    with torch.inference_mode():
        # Get the embeddings, given the generated indices that are on the 'priors' tensor
        z = vqvae_model.vector_quantization.quantize(priors) # [64, H_W_PRIORS, H_W_PRIORS, EMBED_DIM]

        z = z.permute(0, 3, 1, 2).contiguous() # [64, EMBED_DIM, H_W_PRIORS, H_W_PRIORS]

        # Pass embeddings through the VQ-VAE decoder
        pred = vqvae_model.decoder(z)

    # Display the generated images

    #generated_samples = np.array(np.clip((pred + 0.25).cpu().numpy(), 0., 1.) * 255, dtype=np.uint8)  # #### COMMENTED
    generated_samples = np.array(np.clip(pred.cpu().numpy(), 0., 1.) * 255, dtype=np.uint8)            # #### REPLACED BY

    generated_samples = generated_samples.reshape(
        8,
        8,
        3,
        config["model_params"]["patch_size"],
        config["model_params"]["patch_size"]
    )
    permuted_samples = generated_samples.transpose(0, 1, 3, 4, 2)

    fig = plt.figure(figsize=(10, 10), constrained_layout=True)
    gs  = fig.add_gridspec(8, 8)
    for n_row in range(8):
        for n_col in range(8):
            f_ax   = fig.add_subplot(gs[n_row, n_col])
            sample = permuted_samples[n_row, n_col]
            f_ax.imshow(sample)
            f_ax.axis("off")

    # ###################################################################
    # 3. Upload the grid of generated images to W&B

    # pred_images = (pred+0.5)                         ############ COMMENTED
    pred_images = torch.clamp(pred, min=0.0, max=1.0)  ############ ALTERED  
    pred_images = (pred_images*255)
    pred_images = pred_images.to('cpu', dtype=torch.uint8)
    grid        = make_grid(pred_images, nrow=8, padding=1, pad_value=1)
    grid        = grid.permute(1, 2, 0)
    grid        = grid .numpy()

    wandb_image = wandb.Image(grid, caption="VQ-VAE generated images")
    wandb.log({"VQ-VAE generated images": wandb_image})

    # Save the grid of images as a PNG file
    SUFFIX = f'{run}'
    SUFFIX = SUFFIX.zfill(4)
    SUFFIX ="_generated_" + SUFFIX
    file_png = f'results/{config["exp_params"]["pixelcnn_save_file"]}{SUFFIX}.png'
    plt.imsave(file_png, grid)

In [ ]:
# Mark the W&B run as finished
wandb.finish()